# 04-Case study：给 GEM5 增加 L2 NextLinePrefetcher

这一节展示如何把一个新的预取部件接入 GEM5。代码已经作为提交 `8eecd224437707c4c5199bcd49ceca4a15ef8e4a` 合入了 XS-GEM5，不需要重新编译。

## 1. 阅读 L2 NextLinePrefetcher 实现

In [ ]:
%%bash
source ../env.sh

COMMIT=8eecd224437707c4c5199bcd49ceca4a15ef8e4a
git -C "$GEM5_HOME" show --stat --oneline "$COMMIT"

printf '\n1) Core next-line algorithm:\n'
git -C "$GEM5_HOME" show "$COMMIT:src/mem/cache/prefetch/next_line.cc" \
  | sed -n '/NextLine::calculatePrefetch/,/^}/p'

printf '\n2) SimObject registration:\n'
git -C "$GEM5_HOME" show "$COMMIT:src/mem/cache/prefetch/Prefetcher.py" \
  | sed -n '489,495p'

printf '\n3) Apply in kmhv3.py:\n'
git -C "$GEM5_HOME" show "$COMMIT:configs/example/kmhv3.py" \
  | sed -n '205,215p' 

主要的改动为三个方面：

+ 核心算法是把当前地址对齐到 cache line，再请求下一条 line。

+ 将 `NextLinePrefetcher` 注册为一个 `QueuedPrefetcher` SimObject；

+ 将 `NextLinePrefetcher` 挂载到 L2 cache 中，并提供一个 `--l2-next-line-only` 开关表示仅开启 `NextLinePrefetcher`

## 2. 使用开启 L2 NextLinePrefetcher 的 XS-GEM5 运行 memcpy

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/04-gem5/04-case-study-prefetcher/l2-next-line"
mkdir -p "$RUN"
rm -rf "$RUN/m5out"
rm -f "$RUN/perfcct.db"
GEM5_BIN=$(get_asset gem5-precompile/gem5.fast)
MEMCPY=$(get_asset workload/gem5-memcpy-riscv64-xs.bin)
export LD_LIBRARY_PATH="$(get_asset gem5-precompile):${LD_LIBRARY_PATH:-}"
export GCBV_REF_SO=$(get_asset gem5-precompile/riscv64-nemu-interpreter-so)
cd "$GEM5_HOME"

# 使用 --l2-next-line-only 参数仅启用 L2 Cache 的 Next-Line Prefetcher
"$GEM5_BIN" --outdir="$RUN/m5out" \
  "$GEM5_HOME/configs/example/kmhv3.py" \
  --raw-cpt --generic-rv-cpt="$MEMCPY" \
  --l2-next-line-only \
  --enable-arch-db --arch-db-dump-lifetime \
  --arch-db-file="$RUN/perfcct.db" \
  2>&1 | tee "$RUN/run.log" >/dev/null

cat "$RUN/run.log" | tail -n 20

`NextLinePrefetcher` 的 memcpy 带宽是 6002.23 MiB/s。

## 3. 与关闭预取的 baseline 进行带宽比较

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "GEM5").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "04-gem5"))

from gem5_tutorial_utils import plot_memcpy_bandwidth
fig, results = plot_memcpy_bandwidth({
    "No prefetch": ROOT / "work/04-gem5/03-perf/no-prefetch/run.log",
    "L2 next-line": ROOT / "work/04-gem5/04-case-study-prefetcher/l2-next-line/run.log",
})
# display(results[["cycles", "payload_mib_s", "speedup"]])
plt.show()

尽管 `NextLinePrefetcher` 不如 kmhv3 的预取器对 memcpy 带宽提升显著，但是能够验证 `NextLinePrefetcher` 设计的有效性。

## 4. 总结

本次 XS-GEM5 tutorial 给大家演示了如何编译和使用 XS-GEM5，并以 memcpy 这个测试为例给大家演示了我们 XS-GEM5 平台的性能分析工具，给大家展示了 kmhv3 预取器和简单的 L2 NextLine 预取器对 memcpy 性能改善的效果。

虽然 memcpy 是一个很简单的小测试，但是性能分析和优化思路是一致的，不管是对复杂的 SPECCPU 程序甚至黑盒应用 trace 进行性能优化，都可以按照从 Top-Down 瓶颈分析、热点基本块分析、关键问题指令执行行为分析、定位优化点，到设计优化方案，并进行验证闭环这样的思路进行。

**XS-GEM5 平台提供了完善的性能分析工具和流程，欢迎大家使用 XS-GEM5 进行更多的研究和探索！**